# 📰 Selenium: находим элементы и собираем новости

**Методы сбора и обработки данных · Selenium, занятие 2**

Сегодня программа научится *читать* страницу: найти нужный кусок и достать из него текст, ссылку и время. В конце будет:
* 📰 парсер главных новостей lenta.ru — **задание LXP**;
* 📂 три рубрики новостей в таблицу — заготовка к КТ;
* 🔐 вход на сайт через форму — заготовка к КТ.

Тренируемся на учебном сайте **«Новости Полигона»**: https://scr1ptg.github.io/mso_data/polygon/
Он не меняется, а каждое задание само проверяет ответ ✅ / ❌.

### Как работать
1. Запустите ячейку **⚙️ Подготовка** (Shift+Enter). Она ставит Chrome и Selenium — около минуты.
2. В заданиях замените `...` своим кодом и запустите ячейку.
3. Браузер в Colab работает **без окна**. Чтобы увидеть страницу, есть `show(driver)` — она покажет скриншот.
4. Откройте полигон **в соседней вкладке** и нажмите **F12**: там удобно смотреть теги и классы.

In [ ]:
#@title ⚙️ Подготовка: запустите эту ячейку первой (около минуты) { display-mode: "form" }
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("/usr/bin/google-chrome"):
    print("Ставим Google Chrome…")
    subprocess.run("wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"
                   " && apt-get -qq update && apt-get -qq install -y /tmp/chrome.deb",
                   shell=True, check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "selenium"], check=True)
if IN_COLAB or not os.path.exists("common.py"):
    subprocess.run(["wget", "-q", "-O", "common.py", "https://raw.githubusercontent.com/Scr1ptG/mso_data/main/selenium-02/practice/common.py"], check=True)

import importlib, common
importlib.reload(common)
from common import start, polygon, check, show, try_selector
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

try:
    driver.quit()   # если ячейку запускают второй раз — закрываем старый браузер
except Exception:
    pass
driver = start()
wait = WebDriverWait(driver, 10)
driver.get(polygon())
print("✅ Браузер запущен. Открыта страница:", driver.title)
show(driver)

> 🆘 **Если что-то сломалось** (ошибка `invalid session id`, браузер «завис», Colab перезапустился) — просто запустите **⚙️ Подготовку** ещё раз.

---
## 1. Страница — это дерево тегов (DOM)

Selenium видит не картинку, а код страницы: теги, вложенные друг в друга. Главные новости полигона устроены так:

```html
<div class="topnews">                                   ← блок (группа)
  <a class="card-big _topnews" href="article.html?id=301">   ← большая карточка
    <h3 class="card-big__title">Цены на видеокарты…</h3>
    <time class="card-big__date">23:55</time>
  </a>
  <a class="card-mini _topnews" href="article.html?id=101">  ← маленькая карточка
    <span class="card-mini__title">Студенты IThub…</span>
    <time class="card-mini__info-item">23:40</time>
  </a>
  …
</div>
```

**Как найти теги самому:** откройте полигон в соседней вкладке → **F12** → **Ctrl+Shift+C** → кликните по новости. В F12 нажмите **Ctrl+F** и вставьте селектор — справа будет счётчик «1 of 8».

| Способ | Пример | Найдёт |
|---|---|---|
| `By.ID` | `"logo"` | `<a id="logo">` |
| `By.NAME` | `"login"` | `<input name="login">` |
| `By.CLASS_NAME` | `"topnews"` | элемент с этим классом (**один** класс, без точки) |
| `By.TAG_NAME` | `"time"` | `<time>` |
| `By.LINK_TEXT` | `"Спорт"` | `<a>Спорт</a>` |
| `By.CSS_SELECTOR` ⭐ | `".topnews a._topnews"` | что угодно |
| `By.XPATH` | `'//a[text()="Спорт"]'` | что угодно, умеет искать по тексту |

`driver.find_element(...)` — **один** элемент (нет — ошибка). `driver.find_elements(...)` — **список** (нет — пустой `[]`).

### ✏️ Задание 1. Открыть полигон

Сохраните в `title` заголовок вкладки — `driver.title`.

In [ ]:
driver.get(polygon("index.html"))

title = ...  # ВАШ КОД

print("Заголовок вкладки:", title)
check("01", title)

### 🔍 Песочница: сколько найдёт селектор?

`try_selector(driver, "селектор")` работает как Ctrl+F в F12: пишет, сколько элементов нашлось, и обводит их красным на скриншоте.

**CSS-шпаргалка:** `time` — тег · `.card-mini` — класс · `#logo` — id · `a._topnews` — тег **и** класс · `.topnews a` — a **внутри** .topnews · `.a, .b` — **или** · `[name="q"]` — по атрибуту. XPath начинается с `/`.

Сначала **угадайте число**, потом запустите. Попробуйте по очереди: `.card-mini`, `.topnews .card-mini`, `.topnews a`, `.card-big__title, .card-mini__title`, `//h2`.

In [ ]:
try_selector(driver, ".card-mini")

### ✏️ Задание 2. Один элемент пятью способами

У элемента есть `.text` — видимый текст и `.get_attribute("href")` — значение атрибута.

In [ ]:
driver.get(polygon("index.html"))

# а) По ID. Текст логотипа <a id="logo">
logo_text = ...  # ВАШ КОД: driver.find_element(By.ID, "...").text
check("02a", logo_text)

# б) По тексту ссылки. Адрес ссылки «Спорт» в меню
sport_href = ...  # ВАШ КОД: By.LINK_TEXT и .get_attribute("href")
check("02b", sport_href)

# в) По имени. У поля поиска name="q" — достаньте атрибут placeholder
search_placeholder = ...  # ВАШ КОД: By.NAME
check("02c", search_placeholder)

# г) CSS: текст <p class="copyright"> внутри <footer>. Селектор "footer .copyright"
footer_text = ...  # ВАШ КОД: By.CSS_SELECTOR
check("02d", footer_text)

# д) XPath: h2, у которого текст ровно «Главные новости». Подсказка: //h2[text()="..."]
h2_text = ...  # ВАШ КОД: By.XPATH
check("02e", h2_text)

<details><summary>💡 Подсказка</summary>

Шаблон для всех пяти: `driver.find_element(By.СПОСОБ, "что ищем")`, в конце `.text` или `.get_attribute("имя")`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
logo_text = driver.find_element(By.ID, "logo").text
sport_href = driver.find_element(By.LINK_TEXT, "Спорт").get_attribute("href")
```
</details>

### ✏️ Задание 3. Много элементов и поиск внутри блока

Классы `card-big` и `card-mini` есть и в «Главных новостях», и в «По рубриках» — на всей странице их 17.
Нам нужны только главные. Поэтому сначала находим **блок**, а потом ищем **внутри него**:
```python
group = driver.find_element(By.CLASS_NAME, "topnews")
cards = group.find_elements(...)     # ищет только внутри group
```

In [ ]:
driver.get(polygon("index.html"))

# а) Найдите блок class="topnews" и посчитайте ссылки <a> только в нём
group = ...  # ВАШ КОД
count_top = ...  # ВАШ КОД: len(group.find_elements(...))
check("03a", count_top)

# б) Сколько в блоке маленьких карточек (class="card-mini")?
count_mini = ...  # ВАШ КОД
check("03b", count_mini)

# в) Соберите заголовки маленьких карточек (<span class="card-mini__title">) в список
titles = []
# ВАШ КОД: цикл for по маленьким карточкам, titles.append(...)

print(*titles, sep="\n")
check("03c", titles)

<details><summary>💡 Подсказка</summary>

`find_elements` возвращает список — у списка нет `.text`. Чтобы достать текст каждого, нужен цикл `for`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
for card in group.find_elements(By.CLASS_NAME, "card-mini"):
    titles.append(card.find_element(By.CLASS_NAME, "...").text)
```
</details>

---
## 2. Схема «группа → каждая новость в группе»

Это и есть задание LXP: *«Используйте нахождение группы новостей + каждой новости в этой группе»*.

```python
group = driver.find_element(By.CLASS_NAME, "topnews")              # 1. блок
cards = group.find_elements(By.CSS_SELECTOR, "a._topnews")         # 2. карточки ВНУТРИ блока
for card in cards:
    link = card.get_attribute("href")                               # 3. ссылка — атрибут самой карточки
    title = card.find_element(...).text                             #    заголовок — внутри карточки
    time = card.find_element(By.TAG_NAME, "time").text              #    время — внутри карточки
```

⚠️ **Ловушка 1.** Первая карточка большая, заголовок в `.card-big__title`. У остальных — в `.card-mini__title`. CSS умеет «или» через запятую: `".card-big__title, .card-mini__title"`.

⚠️ **Ловушка 2.** В XPath `card.find_element(By.XPATH, "//time")` ищет по **всей странице** — у всех новостей будет одно время. Внутри карточки пишут с точкой: `".//time"`.

### ✏️ Задание 4. Главные новости полигона: ссылка, заголовок, время

Соберите список словарей `{"link": ..., "title": ..., "time": ...}` по всем 8 главным новостям.

In [ ]:
driver.get(polygon("index.html"))

group = ...  # ВАШ КОД: блок главных новостей
cards = []   # ВАШ КОД: все карточки внутри group

news = []
for card in cards:
    link = ...   # ВАШ КОД
    title = ...  # ВАШ КОД
    time = ...   # ВАШ КОД
    news.append({"link": link, "title": title, "time": time})

for n in news:
    print(n["time"], n["title"], n["link"], sep=" | ")
check("04", news)
show(driver, cards)   # найденные карточки обведены красным

<details><summary>💡 Подсказка</summary>

Карточки — это `a` с классом `_topnews`: селектор `"a._topnews"`. Ссылка лежит в атрибуте `href` самой карточки.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
cards = group.find_elements(By.CSS_SELECTOR, "a._topnews")
...
    link = card.get_attribute("href")
    title = card.find_element(By.CSS_SELECTOR, ".card-big__title, .card-mini__title").text
    time = card.find_element(By.TAG_NAME, "time").text
```
</details>

### ✏️ Задание 5. lenta.ru — сдать в LXP

**Задание LXP:** «Напишите парсер, который загрузит главные новости с главной страницы lenta.ru». Вывести ссылку, заголовок и время.

Код из задания 4 почти подходит. Откройте https://lenta.ru в соседней вкладке, нажмите F12 и проверьте сами: как называется блок главных новостей, какой общий класс у карточек, где заголовок и время. Песочница `try_selector` тоже работает на lenta.ru.

Сайт настоящий: разметка может поменяться. Если код перестал работать — снова F12.

In [ ]:
driver.get("https://lenta.ru/")
show(driver)

news = []
# ВАШ КОД

for n in news:
    print(n["time"], n["title"], n["link"], sep=" | ")
check("05", news)

<details><summary>💡 Подсказка</summary>

Попробуйте в песочнице: `try_selector(driver, ".topnews a")`. Сколько нашлось? Совпадает с тем, что видно на скриншоте?
</details>

---
## 3. Ожидания: кнопка «Показать ещё»

На странице рубрики сразу видно 6 новостей. Остальные 6 появятся после кнопки **«Показать ещё»** — и не мгновенно, а через полторы секунды (данные идут по сети).

Если сразу после `click()` посчитать карточки — получится 6. Нужно **подождать**:
* `time.sleep(3)` — работает, но ждёт всегда 3 секунды, даже если всё уже загрузилось;
* `WebDriverWait` — ждёт ровно сколько нужно, но не дольше 10 секунд:

```python
wait.until(EC.invisibility_of_element_located((By.ID, "loadmore")))   # двойные скобки!
```
`wait` уже создан в ячейке «Подготовка».

### ✏️ Задание 6. Три рубрики → таблица (заготовка к КТ)

Рубрики: `science`, `sport`, `economics`, адрес — `polygon(f"rubric.html?r={rubric}")`.
Карточки рубрики — `a._longgrid` внутри `.rubric-page__container`. В каждой рубрике должно быть 12 новостей, всего 36.

In [ ]:
import pandas as pd

rows = []
for rubric in ["science", "sport", "economics"]:
    driver.get(polygon(f"rubric.html?r={rubric}"))
    # ВАШ КОД:
    # 1. найти кнопку id="loadmore" и нажать
    # 2. подождать, пока она исчезнет
    # 3. по каждой карточке: rows.append({"rubric": rubric, "link": ..., "title": ..., "time": ...})
    print(rubric, "— собрано всего:", len(rows))

check("06", rows)
table = pd.DataFrame(rows)
table.to_csv("rubrics.csv", index=False, sep=";", encoding="utf-8-sig")   # файл слева, в папке 📁
table

<details><summary>💡 Подсказка</summary>

Если в проверке «только 6 новостей» — вы не дождались. После `click()` нужна строчка с `wait.until(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
driver.find_element(By.ID, "loadmore").click()
wait.until(EC.invisibility_of_element_located((By.ID, "loadmore")))
for card in driver.find_elements(By.CSS_SELECTOR, ".rubric-page__container a._longgrid"):
    rows.append({"rubric": rubric, "link": ..., "title": ..., "time": ...})
```
</details>

---
## 4. Формы

```python
field = driver.find_element(By.NAME, "login")
field.clear()                 # очистить
field.send_keys("student")    # напечатать
driver.find_element(By.ID, "submit").click()

Select(driver.find_element(By.ID, "op")).select_by_visible_text("умножить")   # выпадающий список
```

### ✏️ Задание 7. Калькулятор

Поля: `<input id="a">`, `<select id="op">`, `<input id="b">`, кнопка `<button id="eq">`, ответ `<span id="result">`.
Ответ появляется не сразу: сначала «считаем…», потом число. Ждём, пока текст станет другим:
```python
wait.until(lambda d: d.find_element(By.ID, "result").text not in ("", "считаем…"))
```

In [ ]:
driver.get(polygon("calc.html"))


def calculate(a, op_text, b):
    """Вводит два числа, выбирает действие, нажимает «=» и возвращает ответ строкой."""
    # ВАШ КОД
    return None


check("07a", calculate(17, "умножить", 23))
check("07b", calculate(10, "разделить", 3))
check("07c", calculate(5, "разделить", 0))
show(driver)

<details><summary>💡 Подсказка</summary>

Не забудьте `clear()` перед `send_keys`, иначе второе число допишется к первому: 17 → 1710.
</details>

### ✏️ Задание 8. Вход на сайт (заготовка к КТ)

Страница входа: `polygon("login.html")`. Учебный логин `student`, пароль `polygon2026` (они написаны на странице).
Поля: `name="login"`, `name="password"`, кнопка `id="submit"`. После входа в кабинете есть секретный код `<b id="secret">`, ошибка входа — в `<p id="error">`.

На настоящем сайте пароль в коде **не пишут**: `from getpass import getpass; password = getpass()`. Для КТ берите тестовый аккаунт, не основной.

In [ ]:
def login(user, password):
    driver.get(polygon("login.html"))
    # ВАШ КОД: заполнить поля и нажать «Войти»


# а) Неверный пароль — дождитесь, пока #error станет видимым, и возьмите .text
login("student", "12345")
error_text = ...  # ВАШ КОД: wait.until(EC.visibility_of_element_located((By.ID, "error"))).text
check("08b", error_text)

# б) Правильный вход — секретный код
login("student", "polygon2026")
secret = ...  # ВАШ КОД
check("08a", secret)
show(driver)

<details><summary>💡 Подсказка</summary>

После `click()` страница меняется не сразу. Ждите нужный элемент через `wait.until(EC.visibility_of_element_located((By.ID, "secret")))` — он же и вернёт элемент.
</details>

### ✏️ Задание 9. Бонус: зайти в каждую новость

1. На главной соберите **ссылки** главных новостей — список строк.
2. Пройдите по списку: `driver.get(ссылка)`, достаньте автора (`.topic-authors__name`), заголовок (`.topic-body__title`) и первый абзац (`.topic-body__content-text`).

⚠️ Если сохранить список *элементов*-карточек и уйти на другую страницу, они «протухнут» — `StaleElementReferenceException`. Сохраняйте строки-ссылки.

In [ ]:
import time

driver.get(polygon("index.html"))
links = []  # ВАШ КОД: ссылки всех 8 главных новостей

articles = []
for link in links:
    driver.get(link)
    # ВАШ КОД: articles.append({"link": link, "title": ..., "author": ..., "text": ...})
    time.sleep(0.5)  # на настоящем сайте — пауза между страницами, чтобы не нагружать сервер

for a in articles:
    print(a.get("author"), "|", a.get("title"))
check("09", articles)

---
## Ошибки — это подсказки

| Ошибка | Что случилось |
|---|---|
| `NoSuchElementException` | селектор ничего не нашёл — проверьте в F12 или `try_selector` |
| `InvalidSelectorException` | опечатка в селекторе; или `By.CLASS_NAME` с двумя классами через пробел |
| `TimeoutException` | `wait` не дождался — условие неверное или элемент не появился |
| `StaleElementReferenceException` | ушли со страницы, а элемент со старой |
| `'list' object has no attribute 'text'` | `.text` у `find_elements` — нужен цикл |
| `invalid session id` | браузер закрылся — запустите **⚙️ Подготовку** ещё раз |

## Вежливый парсинг
* Паузы между страницами: `time.sleep(0.5–1)`.
* Смотрите `robots.txt` сайта.
* Не собираем персональные данные. Входим только в свои — тестовые — аккаунты.

In [ ]:
driver.quit()   # закрыть браузер в конце работы
print("Браузер закрыт")